Url Connection & Data Cleaning

In [54]:
import io
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv
import openmeteo_requests
import requests_cache
from urllib3.util import Retry
from retry import retry
from requests.adapters import HTTPAdapter

load_dotenv()
nasa_api = os.getenv("NASA_MAP_KEY")
source = "VIIRS_NOAA20_NRT"
day_range = 5

top_10_locations = {
  "location_1": {"name": "Amazon Rainforest", "continent": "South America", "west_lon": -80.0, "south_lat": -5.0, "east_lon": -50.0, "north_lat": 5.0},
  "location_2": {"name": "California", "continent": "North America", "west_lon": -124.5, "south_lat": 32.5, "east_lon": -114.0, "north_lat": 42.0},
  "location_3": {"name": "Siberian Taiga", "continent": "Russia", "west_lon": 60.0, "south_lat": 50.0, "east_lon": 140.0, "north_lat": 70.0},
  "location_4": {"name": "New South Wales", "continent": "Austrailia", "west_lon": 141.0, "south_lat": -37.5, "east_lon": 154.0, "north_lat": -28.0},
  "location_5": {"name": "Pantanal", "continent": "Brazil", "west_lon": -62.0, "south_lat": -22.0, "east_lon": -55.0, "north_lat": -16.0},
  "location_6": {"name": "Alberta", "continent": "Canada", "west_lon": -120.0, "south_lat": 49.0, "east_lon": -110.0, "north_lat": 60.0},
  "location_7": {"name": "Mediterranean Basin", "continent": "Southern Europe", "west_lon": -10.0, "south_lat": 30.0, "east_lon": 40.0, "north_lat": 46.0},
  "location_8": {"name": "Indonesia", "continent": "Southeast Asia", "west_lon": 95.0, "south_lat": -11.0, "east_lon": 141.0, "north_lat": 6.0},
  "location_9": {"name": "Greece", "continent": "Southern Europe", "west_lon": 19.0, "south_lat": 34.0, "east_lon": 28.0, "north_lat": 42.0},
  "location_10": {"name": "Algeria", "continent": "North Africa", "west_lon": -9.0, "south_lat": 19.0, "east_lon": 12.0, "north_lat": 37.0}
}

dataframe_list = []
master_df = None

for location_key, location_data in top_10_locations.items():
  west_lon = location_data['west_lon']
  south_lat = location_data['south_lat']
  east_lon = location_data['east_lon']
  north_lat = location_data['north_lat']
  name = location_data['name']

  data_base_url = "https://firms.modaps.eosdis.nasa.gov/api/area"
  prediction_url = f"{data_base_url}/csv/{nasa_api}/{source}/{west_lon},{south_lat},{east_lon},{north_lat}/{day_range}"

  try: 
    response = requests.get(prediction_url)
    response.raise_for_status() 
    
    if "No data available" in response.text or len(response.text.strip()) == 0:
      print(f"No active fires found in {name} for this time range.")
      continue
    
    temp_df = pd.read_csv(io.StringIO(response.text))
    temp_df["location_name"] = name

    dataframe_list.append(temp_df)
    print(f"Successfully added {len(temp_df)} fire points from {name}.")

  except requests.exceptions.HTTPError as e:
    print(f"Error fetching {name}: {e}")
  
  time.sleep(0.5)

if dataframe_list:
    master_df = pd.concat(dataframe_list, ignore_index=True)
    master_df = master_df.drop_duplicates(subset=['latitude', 'longitude', 'bright_ti4', 'acq_date', 'acq_time'])
    print("\nHigh-capacity local aggregation complete!")
    print(f"Combined Dataset Matrix Size: {len(master_df)} Unique Operational Fire Observations")
else:
    print("\nDataset compilation failed. Check your network or API permissions.")
    raise SystemExit

Successfully added 139 fire points from Amazon Rainforest.
Successfully added 345 fire points from California.
Successfully added 2684 fire points from Siberian Taiga.
Successfully added 360 fire points from New South Wales.
Successfully added 1444 fire points from Pantanal.
Successfully added 143 fire points from Alberta.
Successfully added 2820 fire points from Mediterranean Basin.
Successfully added 1632 fire points from Indonesia.
Successfully added 125 fire points from Greece.
Successfully added 1669 fire points from Algeria.

High-capacity local aggregation complete!
Combined Dataset Matrix Size: 10037 Unique Operational Fire Observations


Data Cleaning

In [55]:
master_df.isnull().values.any() # No null values
master_df.duplicated().any() # No duplicated values
master_df.isna().sum() # No na values

latitude         0
longitude        0
bright_ti4       0
scan             0
track            0
acq_date         0
acq_time         0
satellite        0
instrument       0
confidence       0
version          0
bright_ti5       0
frp              0
daynight         0
location_name    0
dtype: int64

Data Exploration

In [56]:
variance_profile = master_df.groupby('location_name')[['frp', 'bright_ti5', 'bright_ti4']].agg(['mean','std'])
print("\n")
print(variance_profile)
print("\n")
prediction_features = master_df[['frp', 'bright_ti5', 'bright_ti4', 'track', 'scan']]
print(prediction_features.head())



                           frp             bright_ti5             bright_ti4  \
                          mean        std        mean        std        mean   
location_name                                                                  
Alberta              10.093217  17.994504  287.044126  11.757644  330.825105   
Algeria               3.191319   4.516091  294.709596   9.975318  319.536617   
Amazon Rainforest     4.988058   3.749840  293.400288   6.209379  332.935899   
California            5.693536   9.374145  296.430464  12.274883  323.813130   
Indonesia             6.860821  12.458587  291.436232   7.077781  329.320680   
Mediterranean Basin   3.581766   6.526683  294.317734   9.499325  320.019525   
New South Wales       6.079472  11.428584  285.090111   6.705412  319.793194   
Pantanal              3.443954   4.545241  290.615159   5.223997  319.427819   
Siberian Taiga        7.607843  12.925977  286.306107  10.675028  325.170861   

                                
    

Creating Risk Level Column

In [57]:
def assign_risk_level(frp_value):
  if frp_value < 5.0:
    return 0
  elif frp_value < 15.0:
    return 1
  elif frp_value < 40.0:
    return 2
  else:
    return 3

master_df['risk_level'] = master_df['frp'].apply(assign_risk_level)

master_df['is_daytime'] = master_df['daynight'].map({'D': 1, 'N': 0})

if master_df['confidence'].dtype == 'O':
    master_df['confidence_num'] = master_df['confidence'].map({'low': 0, 'nominal': 1, 'high': 2})
else:
    master_df['confidence_num'] = master_df['confidence']

# Feature Engineering
master_df['pixel_area'] = master_df['scan'] * master_df['track']
master_df['frp_density'] = master_df['frp'] / master_df['pixel_area']
master_df['thermal_diff'] = master_df['bright_ti4'] / master_df['bright_ti5']
master_df['thermal_ratio'] = master_df['bright_ti4'] / master_df['bright_ti5']
master_df['is_daytime'] = master_df['is_daytime'].fillna(0).astype(int)
master_df['confidence_num'] = pd.to_numeric(master_df['confidence_num'], errors='coerce').fillna(0).astype(float)


Prediction: Fire Radiative Power & Bright Fire Spots

In [58]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

X = master_df[['track', 'scan', 'bright_ti5', 'bright_ti4', 'latitude', 'longitude', 'is_daytime', 'confidence_num', 'pixel_area', 'thermal_diff', 'thermal_ratio']]
y = master_df['risk_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state = 42)

# Printing the shapes
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}, y_train: {y_train.shape}, y_test: {y_test.shape}")


# Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=120, criterion="log_loss", random_state=42)
rf_model.fit(X_train, y_train, sample_weight=None)
rf_predictions = rf_model.predict(X_test)

# Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(n_estimators=100, random_state = 42, verbose=0)
gb_model.fit(X_train, y_train, sample_weight=None)
gb_predictions = gb_model.predict(X_test)
# Add a new Dataframe to see the results clearly
test_results_df = X_test.copy()
test_results_df['True_Risk_Level (Random Forest)'] = y_test
test_results_df['Predicted_Risk_level (Random Forest)'] = rf_predictions
test_results_df['True_Risk_Level (Gradient Boosting)'] = y_test
test_results_df['Predicted_Risk_Level (Gradient Boosting)'] = gb_predictions

test_results_df.head(50)

X_train: (8029, 11), X_test: (2008, 11), y_train: (8029,), y_test: (2008,)


,track,scan,bright_ti5,bright_ti4,latitude,longitude,is_daytime,confidence_num,pixel_area,thermal_diff,thermal_ratio,True_Risk_Level (Random Forest),Predicted_Risk_level (Random Forest),True_Risk_Level (Gradient Boosting),Predicted_Risk_Level (Gradient Boosting)
10291,0.45,0.41,287.37,298.31,28.89169,9.96962,0,0.0,0.1845,1.038069,1.038069,0,0,0,0
6618,0.37,0.40,280.38,308.50,43.37751,-6.97089,0,0.0,0.1480,1.100292,1.100292,0,0,0,0
4441,0.47,0.46,290.32,308.85,-16.52453,-59.79142,0,0.0,0.2162,1.063826,1.063826,0,0,0,0
7500,0.37,0.41,313.24,339.29,35.15380,-3.01133,1,0.0,0.1517,1.083163,1.083163,0,0,0,0
9242,0.53,0.60,292.69,325.02,3.86249,96.44469,1,0.0,0.3180,1.110458,1.110458,1,0,1,0
6442,0.57,0.36,287.97,306.89,36.76297,7.30639,0,0.0,0.2052,1.065701,1.065701,0,0,0,0
7808,0.37,0.41,292.49,306.08,32.92993,3.23607,0,0.0,0.1517,1.046463,1.046463,0,0,0,0
761,0.47,0.45,285.10,332.38,66.73126,80.42147,1,0.0,0.2115,1.165837,1.165837,0,0,0,0
1107,0.40,0.49,281.60,327.52,61.94978,68.46613,0,0.0,0.1960,1.163068,1.163068,1,0,1,0
1650,0.44,0.40,282.58,300.74,55.36950,86.04953,0,0.0,0.1760,1.064265,1.064265,0,0,0,0


Accuracy Score: How good were the model's guesses?

In [59]:
from sklearn.metrics import accuracy_score as acc
from sklearn.metrics import precision_score as pre
from sklearn.metrics import recall_score as rc
from sklearn.metrics import f1_score as f1
from sklearn.metrics import classification_report as cl_report
rf_acc_score = acc(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'])
rf_pre_score = pre(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')
rf_rec_score = rc(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')
rf_f1_score = f1(test_results_df['True_Risk_Level (Random Forest)'], test_results_df['Predicted_Risk_level (Random Forest)'], average='weighted')

gb_acc_score = acc(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'])
gb_pre_score = pre( test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
gb_rec_score = rc(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
gb_f1_score = f1(test_results_df['True_Risk_Level (Gradient Boosting)'], test_results_df['Predicted_Risk_Level (Gradient Boosting)'], average='weighted')
print(f" Accuracy Score: Random Forest --> {rf_acc_score}\n Precision Score: Random Forest --> {rf_pre_score}\n Recall Score: Random Forest --> {rf_rec_score}\n F1 Score: Random Forest --> {rf_f1_score}")
print("\n")
print(f" Accuracy Score: Gradient Boosting --> {gb_acc_score}\n Precision Score: Gradient Boosting --> {gb_pre_score}\n Recall Score: Gradient Boosting --> {gb_rec_score}\n F1 Score: Gradient Boosting --> {gb_f1_score}")
print(cl_report(
    test_results_df['True_Risk_Level (Gradient Boosting)'], 
    test_results_df['Predicted_Risk_Level (Gradient Boosting)'], 
    target_names=['Low Risk (0)', 'Moderate Risk (1)', 'High Risk (2)', 'Extreme Risk (3)']
))


 Accuracy Score: Random Forest --> 0.8172310756972112
 Precision Score: Random Forest --> 0.8019107849812157
 Recall Score: Random Forest --> 0.8172310756972112
 F1 Score: Random Forest --> 0.8062213352871094


 Accuracy Score: Gradient Boosting --> 0.8346613545816733
 Precision Score: Gradient Boosting --> 0.8230472714498545
 Recall Score: Gradient Boosting --> 0.8346613545816733
 F1 Score: Gradient Boosting --> 0.8240813466092519
                   precision    recall  f1-score   support

     Low Risk (0)       0.90      0.93      0.92      1441
Moderate Risk (1)       0.65      0.69      0.67       442
    High Risk (2)       0.50      0.17      0.25       102
 Extreme Risk (3)       0.53      0.35      0.42        23

         accuracy                           0.83      2008
        macro avg       0.65      0.54      0.57      2008
     weighted avg       0.82      0.83      0.82      2008



Prediction: Only 83-85% F1 Score, let's test XG Gradient Boosting

In [60]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
xgb_model = XGBClassifier(objective='multi:softprob',
    num_class=4,
    learning_rate=0.15,     
    max_depth=7,            
    min_child_weight=1,   
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
xgb_preds = xgb_model.predict(X_test)
test_results_df['True_Risk_Level (XG Gradient Boosting)'] = y_test
test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'] = xgb_preds
xgb_acc_score = acc(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'])
xgb_pre_score = pre( test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')
xgb_rec_score = rc(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')
xgb_f1_score = f1(test_results_df['True_Risk_Level (XG Gradient Boosting)'], test_results_df['Predicted_Risk_Level (XG Gradient Boosting)'], average='weighted')
print(f" Accuracy Score: XG Gradient Boosting --> {xgb_acc_score}\n Precision Score: XG Gradient Boosting--> {xgb_pre_score}\n Recall Score: XG Gradient Boosting --> {xgb_rec_score}\n F1 Score: XG Gradient Boosting --> {xgb_f1_score}")
print(classification_report(y_test, xgb_preds, target_names=['Low', 'Moderate', 'High', 'Extreme']))

# The results weren't as I expected, so I will now try GridSearch to find the best settings
xgb_param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    random_state=42
)

xgb_grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=xgb_param_grid,
    scoring='f1_macro',
    cv=3,
    verbose=1,
    n_jobs=-1
)

xgb_grid.fit(X_train, y_train, sample_weight=sample_weights)
best_xgb = xgb_grid.best_estimator_
print(f"\nWinning XGBoost Settings: {xgb_grid.best_params_}")

tuned_xgb_preds = best_xgb.predict(X_test)
print("\nTUNED XGBOOST SCORECARD:")
print(classification_report(y_test, tuned_xgb_preds, target_names=['Low', 'Moderate', 'High', 'Extreme']))

 Accuracy Score: XG Gradient Boosting --> 0.8137450199203188
 Precision Score: XG Gradient Boosting--> 0.8284837642608588
 Recall Score: XG Gradient Boosting --> 0.8137450199203188
 F1 Score: XG Gradient Boosting --> 0.8193549078687613
              precision    recall  f1-score   support

         Low       0.93      0.88      0.90      1441
    Moderate       0.61      0.71      0.65       442
        High       0.40      0.39      0.40       102
     Extreme       0.42      0.61      0.50        23

    accuracy                           0.81      2008
   macro avg       0.59      0.65      0.61      2008
weighted avg       0.83      0.81      0.82      2008

Fitting 3 folds for each of 36 candidates, totalling 108 fits

Winning XGBoost Settings: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 4, 'subsample': 1.0}

TUNED XGBOOST SCORECARD:
              precision    recall  f1-score   support

         Low       0.94      0.84      0.89      1441
    Moderate       0.57

Using JobLib to post the results and cache into memory back to the client

In [61]:
import joblib
from joblib import Memory

joblib.dump(best_xgb, "best_xgb.pkl")
print("Model was saved as a physical file...")
cache_dir = "./wildfire_predictions"
memory = Memory(cache_dir, verbose=1)

@memory.cache
def predict_results(X_matrix):
  print("Computing predictions from user coordinates...")
  return best_xgb.predict(X_matrix)

preds_first = predict_results(X_test)
preds_second = predict_results(X_test)


Model was saved as a physical file...
________________________________________________________________________________
[Memory] Calling __main__--var-folders-c9-_k9zdr3d2958dzv4qjrpmj540000gn-T-ipykernel-3814925403.predict_results...
predict_results(       track  scan  bright_ti5  bright_ti4  latitude  longitude  is_daytime  \
10291   0.45  0.41      287.37      298.31  28.89169    9.96962           0   
6618    0.37  0.40      280.38      308.50  43.37751   -6.97089           0   
4441    0.47  0.46      290.32      308.85 -16.52453  -59.79142           0   
7500    0.37  0.41      313.24      339.29  35.15380   -3.01133           1   
9242    0.53  0.60      292.69      325.02   3.86249   96.44469           1   
6442    0.57  0.36      287.97      306.89  36.76297    7.30639           0   
7808    0.37  0.41      292.49      306.08  32.92993    3.23607           0   
761     0.47  0.45      285.10      332.38  66.73126   80.42147     ...)
Computing predictions from user coordinates..